In [1]:
# setting up the notebook

!pip install transformers torch -q
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
from google.colab import files

# select scoring_input.csv
uploaded = files.upload()
import pandas as pd
df = pd.read_csv('scoring_input.csv', keep_default_na = False)
print('rows to score:', len(df))

Saving scoring_input.csv to scoring_input (1).csv
rows to score: 2700


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# fetching model
NLI_MODEL = 'MilosKosRad/TextualEntailment_DeBERTa_preprocessedSciFACT'
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)

# setting to GPU
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to('cuda')
model.eval()

# index 1 = support
SUPPORT_INDEX = 1

# sanity check
print('model on GPU')

config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

model on GPU


In [4]:
import torch

# function to quickly score the data in batches (32)
def score_batch(passages, claims, batch_size = 32):
    scores = []
    for start in range(0, len(passages), batch_size):
        p = passages[start:start+batch_size]
        c = claims[start:start+batch_size]
        inputs = tokenizer(p, c, return_tensors='pt', truncation = True,
                           max_length = 512, padding = True).to('cuda')
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim = -1)
        scores.extend(probs[:, SUPPORT_INDEX].cpu().tolist())
        print(f'  scored {min(start+batch_size, len(passages))} / {len(passages)}')
    return scores

df['support_score'] = score_batch(df['passage'].tolist(), df['claim'].tolist())
print('done scoring')

  scored 32 / 2700
  scored 64 / 2700
  scored 96 / 2700
  scored 128 / 2700
  scored 160 / 2700
  scored 192 / 2700
  scored 224 / 2700
  scored 256 / 2700
  scored 288 / 2700
  scored 320 / 2700
  scored 352 / 2700
  scored 384 / 2700
  scored 416 / 2700
  scored 448 / 2700
  scored 480 / 2700
  scored 512 / 2700
  scored 544 / 2700
  scored 576 / 2700
  scored 608 / 2700
  scored 640 / 2700
  scored 672 / 2700
  scored 704 / 2700
  scored 736 / 2700
  scored 768 / 2700
  scored 800 / 2700
  scored 832 / 2700
  scored 864 / 2700
  scored 896 / 2700
  scored 928 / 2700
  scored 960 / 2700
  scored 992 / 2700
  scored 1024 / 2700
  scored 1056 / 2700
  scored 1088 / 2700
  scored 1120 / 2700
  scored 1152 / 2700
  scored 1184 / 2700
  scored 1216 / 2700
  scored 1248 / 2700
  scored 1280 / 2700
  scored 1312 / 2700
  scored 1344 / 2700
  scored 1376 / 2700
  scored 1408 / 2700
  scored 1440 / 2700
  scored 1472 / 2700
  scored 1504 / 2700
  scored 1536 / 2700
  scored 1568 / 2700
  sco

In [5]:
# downloading the scores
df[['src_pmid', 'passage_pmid', 'role', 'support_score']].to_csv('scores.csv', index = False)
from google.colab import files

# saving file
files.download('scores.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>